# Amazon Polarity shortcut analysis

This notebook reproduces the non-PCA IMDB experiments on Amazon Polarity. The IMDB-specific numeric/non-numeric split is replaced with an Amazon-specific explicit rating-proxy/non-rating-proxy split. A separate final section analyzes genuinely contiguous salient bigrams and trigrams.


### The three splits

Amazon Polarity originally provides:
- Official train set: 3,600,000 reviews
- Official test set:    400,000 reviews

We split the official training set using: DATA_SPLIT_SEED = 1 into:
train_data:       3,550,000 reviews
validation_data:     50,000 reviews
test_data:          400,000 reviews

**train_data**

Used to update the model weights during fine-tuning.
train_data = dataset["train"]

**validation_data**

Taken out of the original training set before fine-tuning. It is used to measure performance during development, but it is not used for gradient updates.
validation_data = dataset["validation"]
Because train_test_split() creates two disjoint partitions, these 50,000 samples are not in train_data.

**test_data**

This is the unchanged official Amazon Polarity test set:
test_data = dataset["test"]
It was not part of the 3.6-million-example training pool and was not used to train the model.


**Predictions are aligned**

The .npz contains predictions for the official test set in its original order:
amazon_polarity_test_with_preds.npz

The notebook now loads the same:
test_data = dataset["test"]

It checks:
len(test_data) == len(test_preds) == 400_000
and also compares the labels and texts. Therefore:
test_preds[i] corresponds to test_data[i]

**What the seeds affects**

DATA_SPLIT_SEED = 1
determines which 50,000 examples are selected as validation. It does not affect the official test set.

MODEL_SEED = 1
controls model-training randomness.

For seeds 1, 2, and 42, keep:
DATA_SPLIT_SEED = 1
and change only MODEL_SEED.

---
## Setting up S1 Vocabulary

Imports

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import ClassLabel, Dataset, DatasetDict, load_dataset, load_from_disk
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.preprocessing import TransactionEncoder
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import BertForSequenceClassification, BertModel, BertTokenizerFast


Load models and tokenizer: fine-tuned Amazon Polarity BERT and pretrained BERT

In [ ]:
# Change only MODEL_SEED when analysing another fine-tuned model.
MODEL_SEED = 1

model_path = Path(f"./polarity_seed_{MODEL_SEED}/model")

if not model_path.exists():
    raise FileNotFoundError(
        f"Fine-tuned model not found at: {model_path.resolve()}"
    )

tokenizer = BertTokenizerFast.from_pretrained(str(model_path))
ft_bert = BertForSequenceClassification.from_pretrained(str(model_path))

pretrained_name = "bert-base-uncased"
pt_bert = BertModel.from_pretrained(pretrained_name)

print("Model seed:", MODEL_SEED)
print("Fine-tuned model:", model_path.resolve())


Load Dataset

In [ ]:
# DATA_SPLIT_SEED stays fixed so all models use the same train/validation split.
DATA_SPLIT_SEED = 1
VALIDATION_SIZE = 50_000

# Shared cache used by seed 1, seed 2, and seed 42 models.
# It is deliberately outside the polarity_seed_* model folders.
PROCESSED_DATASET_PATH = Path(
    f"./amazon_polarity_processed_split_seed_{DATA_SPLIT_SEED}"
)


def create_text_column(batch):
    """Combine title and review body into one text field."""
    texts = []

    for title, content in zip(batch["title"], batch["content"]):
        title = "" if title is None else str(title).strip()
        content = "" if content is None else str(content).strip()

        text = " ".join(
            part for part in (title, content) if part
        )
        texts.append(text)

    return {"text": texts}


if PROCESSED_DATASET_PATH.exists():
    print("Loading cached processed Amazon Polarity dataset...")
    dataset = load_from_disk(str(PROCESSED_DATASET_PATH))

else:
    print("No processed cache found.")
    print("Downloading and preprocessing Amazon Polarity once...")

    raw_dataset = load_dataset("fancyzhx/amazon_polarity")
    raw_train = raw_dataset["train"]

    # Stratified splitting requires a ClassLabel feature.
    if not isinstance(raw_train.features["label"], ClassLabel):
        raw_train = raw_train.class_encode_column("label")

    train_validation_split = raw_train.train_test_split(
        test_size=VALIDATION_SIZE,
        stratify_by_column="label",
        seed=DATA_SPLIT_SEED,
    )

    dataset = DatasetDict({
        "train": train_validation_split["train"],
        "validation": train_validation_split["test"],
        "test": raw_dataset["test"],
    })

    # After this mapping, every split contains only text and label.
    dataset = dataset.map(
        create_text_column,
        batched=True,
        remove_columns=["title", "content"],
        desc="Creating combined review text",
    )

    dataset.save_to_disk(str(PROCESSED_DATASET_PATH))
    print("Processed dataset saved to:")
    print(PROCESSED_DATASET_PATH.resolve())


# Safety checks against accidentally loading an incompatible cache.
expected_splits = {"train", "validation", "test"}
if set(dataset.keys()) != expected_splits:
    raise ValueError(f"Unexpected dataset splits: {dataset.keys()}")

for split_name in ("train", "validation", "test"):
    if set(dataset[split_name].column_names) != {"text", "label"}:
        raise ValueError(
            f"Unexpected columns in {split_name}: "
            f"{dataset[split_name].column_names}"
        )

print(dataset)
print("Data split seed:", DATA_SPLIT_SEED)
print("Processed dataset path:", PROCESSED_DATASET_PATH.resolve())


In [ ]:
train_data = dataset["train"]
validation_data = dataset["validation"]
official_test_data = dataset["test"]

# This notebook reproduces the test-set shortcut analysis used in the other notebooks.
pipeline_data = official_test_data
test_data = official_test_data

assert len(train_data) == 3_550_000
assert len(validation_data) == 50_000
assert len(test_data) == 400_000

print("Training examples:", len(train_data))
print("Validation examples:", len(validation_data))
print("Pipeline/test examples:", len(test_data))


## Set up device

In [ ]:
# Device setup
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

pt_bert = pt_bert.to(device)
ft_bert = ft_bert.to(device)

pt_bert.eval()
ft_bert.eval()

print("Device:", device)


Method used to extract Saliency.

Source: [Understanding Text Classification Data and Models Using Aggregated Input Salience](https://arxiv.org/abs/2211.05485)

In [ ]:
def grad_l2_saliency(
    text: str,
    model,
    tokenizer,
    device,
    target_class: int | None = None,
    max_length: int = 256
):
    """
    Compute Grad-L2 saliency for one input text.

    Returns:
        tokens: list[str]          # wordpiece tokens incl. special tokens
        saliency: np.ndarray       # shape (seq_len,)
        input_ids: np.ndarray      # token ids (same length as saliency)
    """
    model.eval()

    # 1) Tokenize
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )
    input_ids = enc["input_ids"].to(device)           # [1, L] L = sequence length
    attention_mask = enc["attention_mask"].to(device) # [1, L]
    token_type_ids = enc.get("token_type_ids")
    if token_type_ids is not None:
        token_type_ids = token_type_ids.to(device)

    # 2) Get word embeddings and make them a leaf tensor with grad
    with torch.no_grad():
        word_embeds = model.bert.embeddings.word_embeddings(input_ids)
    word_embeds = word_embeds.detach().requires_grad_(True)  # [1, L, H]

    # 3) Forward pass using inputs_embeds so we can backprop to embeddings
    model.zero_grad()
    outputs = model(
        inputs_embeds=word_embeds, #pass embeddings instead of input_ids
        attention_mask=attention_mask,
        token_type_ids=token_type_ids
    )
    logits = outputs.logits  # [1, num_labels]

    # 4) Choose the logit to explain: pick predicted label by default
    if target_class is None:
        target_class = logits.argmax(dim=-1).item()  # predicted label

    # using the predicted class as scalar for autograd
    score = logits[0, target_class] # standard tensor indexing: tensor[row_index, column_index]

    # 5) Backward: d score / d word_embeds # computes gradients of score with respect to all tensors that have requires_grad=True in the computation graph.
    score.backward()

    grads = word_embeds.grad  # [1, L, H] store gradient 

    # 6) L2 norm over hidden dimension -> one score per token
    saliency = torch.norm(grads, p=2, dim=-1).squeeze(0)  # [1,L,H ] -> [1,L] -> [L] collapsing H

    # 7) Move to CPU / numpy
    saliency = saliency.detach().cpu().numpy()
    input_ids_cpu = input_ids.squeeze(0).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids_cpu)

    return tokens, saliency, input_ids_cpu


Predict Test Set in preparation for S1 Vocabulary

In [ ]:
def predict_labels(
    dataset_split,
    model,
    tokenizer,
    device,
    batch_size=32,
    max_length=256,
):
    model.eval()

    loader = DataLoader(
        dataset_split,
        batch_size=batch_size,
        shuffle=False,
    )

    all_preds = []

    with torch.no_grad():
        for batch in loader:
            texts = batch["text"]

            enc = tokenizer(
                list(texts),
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_length,
            ).to(device)

            logits = model(**enc).logits
            preds = logits.argmax(dim=-1).cpu()
            all_preds.extend(preds.tolist())

    return np.asarray(all_preds, dtype=np.int64)


# Existing predictions from the fine-tuning notebook.
npz_path = Path("amazon_polarity_test_with_preds.npz")

# Seed-specific fallback if predictions ever need to be recomputed.
npy_path = Path(f"test_preds_seed_{MODEL_SEED}.npy")


if npz_path.exists():
    saved_data = np.load(npz_path, allow_pickle=True)
    test_preds = saved_data["pred"].astype(np.int64)

    # Verify row-level alignment with the official Amazon Polarity test split.
    assert len(test_data) == len(test_preds) == 400_000
    assert np.array_equal(
        np.asarray(test_data["label"]),
        saved_data["label"],
    )

    if "seed" in saved_data.files:
        prediction_seed = int(
            np.asarray(saved_data["seed"]).reshape(-1)[0]
        )
        assert prediction_seed == MODEL_SEED, (
            f"The NPZ belongs to model seed {prediction_seed}, "
            f"but MODEL_SEED is {MODEL_SEED}."
        )

    if "text" in saved_data.files:
        chunk_size = 10_000

        for start in range(0, len(test_data), chunk_size):
            end = min(start + chunk_size, len(test_data))

            dataset_text = np.asarray(
                test_data[start:end]["text"],
                dtype=object,
            )
            saved_text = saved_data["text"][start:end]

            assert np.array_equal(dataset_text, saved_text)

    print("Loaded aligned predictions from:", npz_path)

elif npy_path.exists():
    test_preds = np.load(npy_path).astype(np.int64)

    assert len(test_data) == len(test_preds) == 400_000
    print("Loaded predictions from:", npy_path)

else:
    test_preds = predict_labels(
        test_data,
        ft_bert,
        tokenizer,
        device,
        max_length=256,
    )

    np.save(npy_path, test_preds)
    print("Generated and saved predictions to:", npy_path)


print("Number of predictions:", len(test_preds))


Function to extract top 5 salient tokens per sentence.

In [ ]:
def topk_vocab_tokens_with_maxpos(input_ids, saliency, top_k=5, exclude_special_ids=None):
    """
    For one sentence:
    - For each vocab token_id, keep the maximum |saliency| value observed in this sentence,
      and the token position where that maximum occurs.
    - Return the top_k token_ids ranked by that max |saliency|.

    Output format (length <= top_k):
      [{"token_id": int, "position": int, "saliency": float}, ...]
    """
    exclude_special_ids = set(exclude_special_ids or [])
    best = {}  # token_id -> (best_abs_saliency, position)

    for pos, (tok_id, s) in enumerate(zip(input_ids, saliency)):
        tok_id = int(tok_id)
        if tok_id in exclude_special_ids:
            continue

        a = abs(float(s))
        if (tok_id not in best) or (a > best[tok_id][0]):
            best[tok_id] = (a, int(pos))

    if not best:
        return []

    items = [(tok_id, a, pos) for tok_id, (a, pos) in best.items()]
    items.sort(key=lambda x: x[1], reverse=True)
    items = items[:top_k]

    return [{"token_id": tok_id, "saliency": a, "position": pos} for tok_id, a, pos in items]

Fill up row per sentence vectors for top-5 tokens:

---

**Build S1-Vocab**

This function converts the first n text examples into their salience-fingerprints (S1 vectors), stacks those into a matrix, normalizes them, and returns both the matrix and the original texts.

Text example -> Row

<pre>
          column dimension → |V| (the r vector)
        ┌──────────────────────────────────────────────┐
 row 1  │ r(1)[0]   r(1)[1]   r(1)[2]   ...   r(1)[V-1]│
 row 2  │ r(2)[0]   r(2)[1]   r(2)[2]   ...   r(2)[V-1]│
   .    │    .         .         .              .      │
 row N  │ r(N)[0]   r(N)[1]   r(N)[2]   ...   r(N)[V-1]│
        └──────────────────────────────────────────────┘
             ↑
             vocabulary-dimension (S1 vector)***
<pre>

In [ ]:
def saliency_to_vocab_vector(input_ids, saliency, vocab_size, top_k=5):
    """
    Implements S1 from the paper: a |V|-dim vector with the
    max |saliency| per vocab item, zeroed except for top-k entries.
    """
    r = np.zeros(vocab_size, dtype=np.float32) # expecting vocab size of bert tokenzier

    for tok_id, s in zip(input_ids, saliency):
        tok_id = int(tok_id)
        r[tok_id] = max(r[tok_id], abs(float(s))) # get max saliency value

    if top_k is not None and top_k < vocab_size:
        # Keep only top-k entries, zero out the rest
        if np.count_nonzero(r) > top_k: # count how many positoons in r are not zero
            top_idx = np.argpartition(r, -top_k)[-top_k:] # Find the indices of the k largest values in r, argsort begin nlogn efficient
            mask = np.zeros_like(r)
            mask[top_idx] = r[top_idx]
            r = mask

    return r


def build_salience_vocab_matrix_from_indices(
    dataset_split,
    indices,
    model,
    tokenizer,
    device,
    top_k=5,
    max_length=256,
):
    """Build a row-normalized vocabulary salience matrix for selected dataset indices.

    For each text, computes token saliency, aggregates the top-k salient tokens into a
    vocabulary-sized vector, and returns the stacked matrix along with the source texts
    and per-example top-k token metadata including raw and normalized saliency scores.
    """
    vocab_size = tokenizer.vocab_size
    R_rows = []
    texts = []
    salient_token_info = [] # save positions

    special_ids = tokenizer.all_special_ids  


    for idx in tqdm(indices):
        text = dataset_split[idx]["text"]
        tokens, sal, ids = grad_l2_saliency(
            text, model, tokenizer, device, max_length=max_length
        )
        r = saliency_to_vocab_vector(ids, sal, vocab_size, top_k=top_k)
        R_rows.append(r)
        texts.append(text)


        # extract token_positions
        topk_vocab = topk_vocab_tokens_with_maxpos(
            ids, sal, top_k=top_k, exclude_special_ids=special_ids
        )
        salient_token_info.append({
            "dataset_idx": idx,
            "text": text,
            "topk_vocab": topk_vocab
        })

    R_vocab = np.stack(R_rows, axis=0)
    # row-wise L2 normalization as in the paper
    norms = np.linalg.norm(R_vocab, axis=1, keepdims=True) + 1e-12
    R_vocab = R_vocab / norms
    #store both raw + normalized saliency per token in salient_token_info
    for row_i, info in enumerate(salient_token_info):
        for item in info["topk_vocab"]:
            tok_id = int(item["token_id"])
            item["saliency_raw"] = float(item["saliency"])              # existing raw abs-saliency
            item["saliency_norm"] = float(R_vocab[row_i, tok_id])       # exact value from r_vocab row

    return R_vocab, texts, salient_token_info

In [ ]:
# Select examples predicted as negative by the current model.
neg_indices = np.flatnonzero(test_preds == 0)

# This controls only which 2,500 test examples are sampled.
# Use 42, 1, and 2 for the three sampling repetitions.
SAMPLE_SEED = 1
SAMPLE_SIZE = 2_500

rng = np.random.default_rng(SAMPLE_SEED)

sample_size = min(SAMPLE_SIZE, len(neg_indices))
neg_sample_indices = rng.choice(
    neg_indices,
    size=sample_size,
    replace=False,
)

R_vocab_5_neg, texts_neg, salient_token_info_neg = (
    build_salience_vocab_matrix_from_indices(
        test_data,
        neg_sample_indices,
        ft_bert,
        tokenizer,
        device,
        top_k=5,
    )
)

print("Model seed:", MODEL_SEED)
print("Sample seed:", SAMPLE_SEED)
print("Negative sample size:", len(neg_sample_indices))


Create S1-Vocab for 2500 positively predicted samples

In [ ]:
# Optional positive-class analysis.
#
# pos_indices = np.flatnonzero(test_preds == 1)
# rng_pos = np.random.default_rng(SAMPLE_SEED)
#
# pos_sample_indices = rng_pos.choice(
#     pos_indices,
#     size=min(SAMPLE_SIZE, len(pos_indices)),
#     replace=False,
# )
#
# R_vocab_5_pos, texts_pos, salient_token_info_pos = (
#     build_salience_vocab_matrix_from_indices(
#         test_data,
#         pos_sample_indices,
#         ft_bert,
#         tokenizer,
#         device,
#         top_k=5,
#     )
# )


## Original method: FP-growth over bags of salient tokens

This reproduces the IMDB pipeline. FP-growth receives unordered sets of salient token IDs. The `ordered_ids` column is added afterward for presentation and hidden-state experiments; these patterns are not yet guaranteed to be contiguous n-grams.


Imports

Setup Transactions and Lookup Table (Salient tokens per Sentence identified with L2-Grad Saliency Method)

In [ ]:
transactions_neg_5 = []
pos_lookup_neg_5 = []   # list of dicts: token_id -> position (for this transaction)

for item in salient_token_info_neg:
    toks = [int(t["token_id"]) for t in item["topk_vocab"]]

    transactions_neg_5.append(toks)
    pos_lookup_neg_5.append({int(t["token_id"]): int(t["position"]) for t in item["topk_vocab"]})

print("num transactions_neg_5:", len(transactions_neg_5))
print("avg len:", np.mean([len(t) for t in transactions_neg_5]))
print(transactions_neg_5[:5])

Running FP-Algorithm

In [ ]:
te = TransactionEncoder() # Creates encoder that converts lists of tokens into a binary matrix (One hot encoded)
X = te.fit(transactions_neg_5).transform(transactions_neg_5) # fit and transform bool matrix to prepare for fp-growth

df = pd.DataFrame(X, columns=te.columns_) # to pandas

freq_itemsets_neg_5 = fpgrowth(
    df,
    min_support=0.002, # minimum frequency of items -> 0.002 = 0.2%
    use_colnames=True, # return column names not indices
    max_len=4 # max item size per pattern
)

freq_itemsets_neg_5["length"] = freq_itemsets_neg_5["itemsets"].apply(len) # add length column

def order_itemset_like_sentence(itemset):
    # pick a transaction and use its token positions as the order reference
    items = [int(x) for x in itemset]

    for tx, pm in zip(transactions_neg_5, pos_lookup_neg_5):
        if all(i in pm for i in items):
            return sorted(items, key=lambda i: pm[i])

    return items


freq_itemsets_neg_5["ordered_ids"] = freq_itemsets_neg_5["itemsets"].apply(order_itemset_like_sentence)
freq_itemsets_neg_5["ordered_tokens"] = freq_itemsets_neg_5["ordered_ids"].apply(lambda ids: tokenizer.convert_ids_to_tokens(ids))

Inspect frequent, high-support salient-token itemsets with length greater than one.

In [ ]:
freq_itemsets_neg_5[freq_itemsets_neg_5["length"] > 1].sort_values("support", ascending=False).head(40)

### Creating negative R_Vocab with the top 10 salient tokens per sentence.

In [ ]:
R_vocab_10_neg, texts_neg_10, salient_token_info_neg_10 = build_salience_vocab_matrix_from_indices(
    test_data,
    neg_sample_indices,
    ft_bert,
    tokenizer,
    device,
    top_k=10, ######################
)

Setup Transactions and Lookup Table (Salient tokens per Sentence identified with L2-Grad Saliency Method)

In [ ]:
transactions_neg_10 = []
pos_lookup_10 = []   # list of dicts: token_id -> position (for this transaction)

for item in salient_token_info_neg_10:
    toks = [int(t["token_id"]) for t in item["topk_vocab"]]

    transactions_neg_10.append(toks)
    pos_lookup_10.append({int(t["token_id"]): int(t["position"]) for t in item["topk_vocab"]})

print("num transactions_neg_10:", len(transactions_neg_10))
print("avg len:", np.mean([len(t) for t in transactions_neg_10]))
print(transactions_neg_10[:5])

Running FP-Algorithm

In [ ]:
te = TransactionEncoder() # Creates encoder that converts lists of tokens into a binary matrix (One hot encoded)
X = te.fit(transactions_neg_10).transform(transactions_neg_10) # fit and transform bool matrix to prepare for fp-growth

df = pd.DataFrame(X, columns=te.columns_) # to pandas

freq_itemsets_neg_10 = fpgrowth(
    df,
    min_support=0.002, # minimum frequency of items -> 0.002 = 0.2%
    use_colnames=True, # return column names not indices
    max_len=4 # max item size per pattern
)

freq_itemsets_neg_10["length"] = freq_itemsets_neg_10["itemsets"].apply(len) # add length column

def order_itemset_like_sentence(itemset):
    # pick a transaction and use its token positions as the order reference
    items = [int(x) for x in itemset]

    for tx, pm in zip(transactions_neg_10, pos_lookup_10):
        if all(i in pm for i in items):
            return sorted(items, key=lambda i: pm[i])

    return items


freq_itemsets_neg_10["ordered_ids"] = freq_itemsets_neg_10["itemsets"].apply(order_itemset_like_sentence)
freq_itemsets_neg_10["ordered_tokens"] = freq_itemsets_neg_10["ordered_ids"].apply(lambda ids: tokenizer.convert_ids_to_tokens(ids))

Inspect frequent salient-token itemsets. Ordering alone does not yet imply contiguity.

In [ ]:
freq_itemsets_neg_10[freq_itemsets_neg_10["length"] > 1].sort_values("support", ascending=False).head(40)

In [ ]:
freq_itemsets_neg_10[freq_itemsets_neg_10["length"] > 2].sort_values("support", ascending=False).head(40)

---
## Experiment 1: Saliency and representation shifts of frequent salient-token itemsets

Add Saliency values per transaction

In [ ]:
transactions_neg_10 = []
pos_lookup_neg_10 = []   # list of dicts: token_id -> position (per sample)
saliency_lookup = []   # list of dicts: token_id -> saliency_norm (per sample)


for item in salient_token_info_neg_10:
    toks = [int(t["token_id"]) for t in item["topk_vocab"]]

    transactions_neg_10.append(toks)
    pos_lookup_neg_10.append({int(t["token_id"]): int(t["position"]) for t in item["topk_vocab"]})

    saliency_lookup.append({int(t["token_id"]): float(t["saliency_norm"]) for t in item["topk_vocab"]})

print("transactions_neg_10:", len(transactions_neg_10))
print("avg transaction length:", np.mean([len(t) for t in transactions_neg_10]))
print(transactions_neg_10[:5])

# Convert transactions_neg_10 to matrix
te = TransactionEncoder()
X = te.fit(transactions_neg_10).transform(transactions_neg_10)
df = pd.DataFrame(X, columns=te.columns_)

print("transaction matrix shape:", df.shape)

#Run FP-Growth
freq_itemsets_neg_10 = fpgrowth(
    df,
    min_support=0.002, # minimum frequency of items -> 0.002 = 0.2%
    use_colnames=True, # return column names not indices
    max_len=4 # max item size per pattern
)

freq_itemsets_neg_10["length"] = freq_itemsets_neg_10["itemsets"].apply(len)

def order_itemset_like_sentence(itemset):
    # pick a transaction and use its token positions as the order reference
    items = [int(x) for x in itemset]

    for tx, pm in zip(transactions_neg_10, pos_lookup_neg_10):
        if all(i in pm for i in items):
            return sorted(items, key=lambda i: pm[i])

    return items

freq_itemsets_neg_10["ordered_ids"] = freq_itemsets_neg_10["itemsets"].apply(order_itemset_like_sentence)
freq_itemsets_neg_10["ordered_tokens"] = freq_itemsets_neg_10["ordered_ids"].apply(
    lambda ids: tokenizer.convert_ids_to_tokens([int(x) for x in ids])
)

# compute saliency only from the tokens in the itemset,
# and only in the supporting transactions_neg_10 of that itemset ===
def pattern_saliency(itemset):
    items = [int(t) for t in itemset]
    scores = []

    for tx, sm in zip(transactions_neg_10, saliency_lookup):
        if all(i in tx for i in items):   # supporting transaction
            for i in items:               # only itemset tokens
                scores.append(sm[i])

    return sum(scores) / len(scores) if scores else 0


freq_itemsets_neg_10["saliency_score"] = freq_itemsets_neg_10["itemsets"].apply(pattern_saliency)
freq_itemsets_neg_10["score"] = freq_itemsets_neg_10["support"] * freq_itemsets_neg_10["saliency_score"]

### Amazon-specific shortcut-candidate grouping

The IMDB notebook compared numeric patterns with non-numeric patterns because
explicit numeric ratings were treated as shortcut candidates.

For Amazon Polarity, the analogous shortcut group is **explicit rating-value
proxies**, for example:

- `1 star`, `2 stars`, `one star`, `two stars`;
- `1-star`, `five-star`;
- `1/5`, `2 out of 5`;
- `rating of 1`, `rated it 2`.

A number by itself is not classified as a shortcut because it may describe a
price, quantity, age, page count, model number, or another legitimate product
attribute. Likewise, `star` or `star rating` without a rating value is not
enough to determine sentiment.


In [ ]:
# Amazon rating values represented as digits or words.
RATING_VALUE_PATTERN = (
    r"(?:[0-5]|zero|one|two|three|four|five|no)"
)

AMAZON_RATING_PROXY_PATTERNS = [
    # 1 star, two stars, five-star, no stars
    re.compile(
        rf"\b{RATING_VALUE_PATTERN}\s*(?:-\s*)?stars?\b",
        flags=re.IGNORECASE,
    ),

    # 1/5, 2 / 5
    re.compile(
        rf"\b{RATING_VALUE_PATTERN}\s*/\s*5\b",
        flags=re.IGNORECASE,
    ),

    # 1 out of 5, two out of five
    re.compile(
        rf"\b{RATING_VALUE_PATTERN}\s+out\s+of\s+(?:5|five)\b",
        flags=re.IGNORECASE,
    ),

    # rating 1, rating: 1, rating of two
    re.compile(
        rf"\brating\s*(?::|=|of)?\s*{RATING_VALUE_PATTERN}\b",
        flags=re.IGNORECASE,
    ),

    # rated 1, rated it two, rated this as 1
    re.compile(
        rf"\brated\s+(?:(?:it|this|product|the\s+product)\s+)?"
        rf"(?:(?:as|at)\s+)?{RATING_VALUE_PATTERN}\b",
        flags=re.IGNORECASE,
    ),
]


def candidate_tokens_to_text(ordered_tokens):
    """
    Convert BERT WordPiece tokens into normalized readable text.
    """
    if isinstance(ordered_tokens, (list, tuple)):
        text = tokenizer.convert_tokens_to_string(
            list(ordered_tokens)
        )
    else:
        text = str(ordered_tokens)

    text = text.lower()
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def is_amazon_rating_proxy(ordered_tokens):
    """
    Return True only for an explicit rating value.

    Examples classified as True:
        1 star
        two stars
        1/5
        one out of five
        rating of 1

    Examples classified as False:
        3 months
        200 pages
        star rating
        poor quality
    """
    text = candidate_tokens_to_text(ordered_tokens)

    return any(
        pattern.search(text)
        for pattern in AMAZON_RATING_PROXY_PATTERNS
    )


def add_amazon_candidate_group(df):
    """
    Add readable text and the Amazon shortcut-candidate group.
    """
    out = df.copy()

    out["candidate_text"] = out["ordered_tokens"].apply(
        candidate_tokens_to_text
    )
    out["is_rating_proxy"] = out["ordered_tokens"].apply(
        is_amazon_rating_proxy
    )
    out["candidate_group"] = np.where(
        out["is_rating_proxy"],
        "rating_proxy",
        "non_rating_proxy",
    )

    return out


In [ ]:
two_grams = (
    freq_itemsets_neg_10[
        freq_itemsets_neg_10["length"] > 1
    ]
    .sort_values("score", ascending=False)
    .head(100)
    .copy()
)

two_grams = add_amazon_candidate_group(two_grams)

print(two_grams["candidate_group"].value_counts())


In [ ]:
two_grams.head(20)

Comparing Saliency and Support of highes support itemsets

In [ ]:
# convert token lists to readable strings
labels = two_grams["ordered_tokens"].apply(lambda x: " ".join(x))

fig, axes = plt.subplots(2, 1, figsize=(18,10))  # 2 rows, 1 column

# first plot
axes[0].bar(labels, two_grams["saliency_score"])
axes[0].set_xlabel("ordered_tokens")
axes[0].set_ylabel("saliency")
axes[0].set_title("Saliency score of top frequent token patterns")
axes[0].tick_params(axis="x", rotation=70)

# second plot (example)
axes[1].bar(labels, two_grams["support"])  # replace with your second data
axes[1].set_xlabel("ordered_tokens")
axes[1].set_ylabel("support")
axes[1].set_title("Support score of top frequent token patterns")
axes[1].tick_params(axis="x", rotation=70)

plt.tight_layout()
plt.show()

Function to extract hidden states for high support tokens.

In [ ]:
@torch.no_grad()
def extract_itemset_hidden_states(
    model,
    tokenizer,
    itemsets_token_ids, # list of lists, e.g. [[101, 202], [55, 77, 88], ..]
    device,
    layers=(11, 12),
    context_mode="isolated",     # "isolated" vs "template"
    template_prefix="This product is",
    batch_size=32,
    num_labels=2,
):
    """
    Returns:
      out: dict layer_idx -> list[np.ndarray of shape (itemset_len, hidden_dim)]
      logits_out: np.ndarray [n_itemsets, num_labels]
    """

    model.eval()

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    if context_mode == "isolated":
        base_ids = []
        token_start = 1
    elif context_mode == "template":
        base_ids = tokenizer(template_prefix, add_special_tokens=False).input_ids
        token_start = 1 + len(base_ids)
    else:
        raise ValueError("context_mode must be 'isolated' or 'template'")

    out = {L: [] for L in layers}
    logits_out = []

    for start in range(0, len(itemsets_token_ids), batch_size):
        batch_itemsets = itemsets_token_ids[start:start + batch_size]

        input_ids_list = []
        token_ranges = []

        for ids in batch_itemsets:
            ids = [int(t) for t in ids]

            full_ids = [cls_id] + base_ids + ids + [sep_id]
            input_ids_list.append(full_ids)

            # positions of the itemset tokens inside the sequence
            token_ranges.append((token_start, token_start + len(ids)))

        max_len = max(len(x) for x in input_ids_list)

        padded_input_ids = []
        attention_masks = []
        token_type_ids = []

        for ids in input_ids_list:
            pad_len = max_len - len(ids)
            padded_input_ids.append(ids + [pad_id] * pad_len)
            attention_masks.append([1] * len(ids) + [0] * pad_len)
            token_type_ids.append([0] * max_len)

        input_ids = torch.tensor(padded_input_ids, dtype=torch.long, device=device)
        attention_mask = torch.tensor(attention_masks, dtype=torch.long, device=device)
        token_type_ids = torch.tensor(token_type_ids, dtype=torch.long, device=device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,
            return_dict=True,
        )

        if hasattr(outputs, "logits"):
            logits_out.append(outputs.logits.detach().cpu().numpy())
        else:
            bsz = input_ids.shape[0]
            logits_out.append(np.full((bsz, num_labels), np.nan, dtype=float))

        hs = outputs.hidden_states

        for b_idx, (s, e) in enumerate(token_ranges):
            for L in layers:
                vecs = hs[L][b_idx, s:e, :].detach().cpu().numpy()   # shape: [itemset_len, hidden_dim]
                out[L].append(vecs)

    logits_out = np.vstack(logits_out)

    return out, logits_out

Run Function for isolated tokens and for tokens within a template

In [ ]:
layers = (5, 8, 11, 12)
itemset_ids = two_grams["ordered_ids"].tolist()

# Pretrained
pt_iso, pt_logits_iso = extract_itemset_hidden_states(
    pt_bert, tokenizer, itemset_ids, device, layers=layers, context_mode="isolated"
)
pt_tpl, pt_logits_tpl = extract_itemset_hidden_states(
    pt_bert, tokenizer, itemset_ids, device, layers=layers, context_mode="template"
)

# Finetuned
ft_iso, ft_logits_iso = extract_itemset_hidden_states(
    ft_bert, tokenizer, itemset_ids, device, layers=layers, context_mode="isolated"
)
ft_tpl, ft_logits_tpl = extract_itemset_hidden_states(
    ft_bert, tokenizer, itemset_ids, device, layers=layers, context_mode="template"
)

Add to dataframe

In [ ]:
for L in layers:
    two_grams[f"pt_iso_l{L}"] = list(pt_iso[L])
    two_grams[f"pt_tpl_l{L}"] = list(pt_tpl[L])
    two_grams[f"ft_iso_l{L}"] = list(ft_iso[L])
    two_grams[f"ft_tpl_l{L}"] = list(ft_tpl[L])

Inspect consine/euclidean distance between mean embeddings l12 from pre trained and finetuned bert per itemset

In [ ]:
import numpy as np

def pool(x):
    x = np.asarray(x)
    return x.mean(axis=0) if x.ndim == 2 else x

def cos_shift(pt, ft):
    a = pool(pt)
    b = pool(ft)
    cos = np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))
    return 1 - cos

def euc_dist(pt, ft):
    a = pool(pt)
    b = pool(ft)
    return np.linalg.norm(a - b)

layers = (5, 8, 11, 12)

top = two_grams.sort_values("support", ascending=False).head(30).copy()

for L in layers:
    top[f"cos_shift_iso_l{L}"] = [
        cos_shift(pt, ft) if pt is not None and ft is not None else None
        for pt, ft in zip(top[f"pt_iso_l{L}"], top[f"ft_iso_l{L}"])
    ]

    top[f"euc_iso_l{L}"] = [
        euc_dist(pt, ft) if pt is not None and ft is not None else None
        for pt, ft in zip(top[f"pt_iso_l{L}"], top[f"ft_iso_l{L}"])
    ]

    top[f"cos_shift_tpl_l{L}"] = [
        cos_shift(pt, ft) if pt is not None and ft is not None else None
        for pt, ft in zip(top[f"pt_tpl_l{L}"], top[f"ft_tpl_l{L}"])
    ]

    top[f"euc_tpl_l{L}"] = [
        euc_dist(pt, ft) if pt is not None and ft is not None else None
        for pt, ft in zip(top[f"pt_tpl_l{L}"], top[f"ft_tpl_l{L}"])
    ]

In [ ]:
# Experiments

layers = (12,11,8,5)

fig, axes = plt.subplots(len(layers), 2, figsize=(14, 4 * len(layers)))
plt.subplots_adjust(wspace=0.6, hspace=0.4)

for i, L in enumerate(layers):
    # --- Cosine ---
    s1 = top.sort_values(f"cos_shift_iso_l{L}", ascending=False)
    axes[i, 0].barh(range(len(s1)), s1[f"cos_shift_iso_l{L}"])
    axes[i, 0].set_yticks(range(len(s1)))
    axes[i, 0].set_yticklabels(s1.ordered_tokens)
    axes[i, 0].set_title(f"Cosine shift (l{L}, iso)")
    axes[i, 0].invert_yaxis()

    # --- Euclidean ---
    s2 = top.sort_values(f"euc_tpl_l{L}", ascending=False)
    axes[i, 1].barh(range(len(s2)), s2[f"euc_tpl_l{L}"])
    axes[i, 1].set_yticks(range(len(s2)))
    axes[i, 1].set_yticklabels(s2.ordered_tokens)
    axes[i, 1].set_title(f"Euclidean distance (l{L}, tpl)")
    axes[i, 1].invert_yaxis()

    # move right labels outward to avoid overlap
    axes[i, 1].yaxis.tick_right()

plt.tight_layout()

## Experiment 2: Inspecting Hidden States: Layer 5,8,11 and 12

- Using freq_itemset with all tokens since we want a larger pool of tokens for neighbourhood analysis

In [ ]:
freq_itemsets_neg_10

In [ ]:
neighbour_analysis_df = (
    freq_itemsets_neg_10[
        freq_itemsets_neg_10["length"] > 0
    ]
    .sort_values("score", ascending=False)
    .copy()
)

neighbour_analysis_df = add_amazon_candidate_group(
    neighbour_analysis_df
)

print(neighbour_analysis_df["candidate_group"].value_counts())


Adding Hidden States

In [ ]:
layers = (5, 8, 11, 12)
itemset_ids = neighbour_analysis_df["ordered_ids"].tolist()

# Pretrained
pt_iso, pt_logits_iso = extract_itemset_hidden_states(
    pt_bert, tokenizer, itemset_ids, device, layers=layers, context_mode="isolated"
)
pt_tpl, pt_logits_tpl = extract_itemset_hidden_states(
    pt_bert, tokenizer, itemset_ids, device, layers=layers, context_mode="template"
)

# Finetuned
ft_iso, ft_logits_iso = extract_itemset_hidden_states(
    ft_bert, tokenizer, itemset_ids, device, layers=layers, context_mode="isolated"
)
ft_tpl, ft_logits_tpl = extract_itemset_hidden_states(
    ft_bert, tokenizer, itemset_ids, device, layers=layers, context_mode="template"
)

# Assign pretrained
neighbour_analysis_df["pt_iso_l5"]  = list(pt_iso[5])
neighbour_analysis_df["pt_iso_l8"]  = list(pt_iso[8])
neighbour_analysis_df["pt_iso_l11"] = list(pt_iso[11])
neighbour_analysis_df["pt_iso_l12"] = list(pt_iso[12])

neighbour_analysis_df["pt_tpl_l5"]  = list(pt_tpl[5])
neighbour_analysis_df["pt_tpl_l8"]  = list(pt_tpl[8])
neighbour_analysis_df["pt_tpl_l11"] = list(pt_tpl[11])
neighbour_analysis_df["pt_tpl_l12"] = list(pt_tpl[12])

# Assign finetuned
neighbour_analysis_df["ft_iso_l5"]  = list(ft_iso[5])
neighbour_analysis_df["ft_iso_l8"]  = list(ft_iso[8])
neighbour_analysis_df["ft_iso_l11"] = list(ft_iso[11])
neighbour_analysis_df["ft_iso_l12"] = list(ft_iso[12])

neighbour_analysis_df["ft_tpl_l5"]  = list(ft_tpl[5])
neighbour_analysis_df["ft_tpl_l8"]  = list(ft_tpl[8])
neighbour_analysis_df["ft_tpl_l11"] = list(ft_tpl[11])
neighbour_analysis_df["ft_tpl_l12"] = list(ft_tpl[12])

# logits
neighbour_analysis_df["ft_iso_logits"] = list(ft_logits_iso)
neighbour_analysis_df["ft_tpl_logits"] = list(ft_logits_tpl)

In [ ]:
neighbour_analysis_df.columns

Building three dfs for nieghbourhood analysis of embeddings of itemsets:
- Inspecting mean of all tokens from one itemset
- Inspecting just the first embedding of an itemset
- Inspecting the sum of all tokens within an itemset

## Experiment 3: Neighbourhood analysis

Aggregating Embeddings

In [ ]:
# embedding columns you want to transform
layers = (5, 8, 11, 12)
types = ("pt", "ft")
contexts = ("iso", "tpl")

embedding_cols = [
    f"{type}_{context}_l{layer}"
    for type in types
    for context in contexts
    for layer in layers
]

# base dataframe
base_cols = [
    "support",
    "length",
    "ordered_ids",
    "ordered_tokens",
    "saliency_score",
    "candidate_text",
    "is_rating_proxy",
    "candidate_group",
]

base_df = neighbour_analysis_df[base_cols + embedding_cols].copy()

def emb_mean(x):
    return np.asarray(x).mean(axis=0)

def emb_first(x):
    return np.asarray(x)[0]

def emb_sum(x):
    return np.asarray(x).sum(axis=0)

# build the 3 result dataframes
df_mean = base_df[base_cols].copy()
df_first_token = base_df[base_cols].copy()
df_sum = base_df[base_cols].copy()

for col in embedding_cols:
    df_mean[col] = base_df[col].apply(emb_mean)
    df_first_token[col] = base_df[col].apply(emb_first)
    df_sum[col] = base_df[col].apply(emb_sum)

# reset index
df_mean = df_mean.reset_index(drop=True)
df_first_token = df_first_token.reset_index(drop=True)
df_sum = df_sum.reset_index(drop=True)

print(df_mean.shape, df_first_token.shape, df_sum.shape)
display(df_mean.head(2))
display(df_first_token.head(2))
display(df_sum.head(2))

Extracting Hidden States for Each token: Layer12

In [ ]:
# extract uniquq token ids
unique_token_ids = sorted(set(tok for ids in df_mean["ordered_ids"] for tok in ids))

#token to strings
id_to_token = {}
for ids, toks in zip(df_mean["ordered_ids"], df_mean["ordered_tokens"]):
    for i, t in zip(ids, toks):
        id_to_token[i] = t

token_df = pd.DataFrame({
    "token_id": unique_token_ids,
    "token": [id_to_token[i] for i in unique_token_ids]
})


In [ ]:
import numpy as np
import torch

@torch.no_grad()
def extract_token_hidden_states(
    model,
    tokenizer,
    token_ids, # list[int]
    device,
    layers=(5, 8, 11, 12),
    context_mode="isolated",    # "isolated" or "template"
    template_prefix="This product is",
    batch_size=64,
):
    """
    Extract single-token embeddings from one BERT model.

    Modes
    -----
    isolated:
        [CLS] token [SEP]

    template:
        [CLS] template_prefix token [SEP]

    Returns
    -------
    dict: layer_idx -> np.ndarray of shape (n_tokens, hidden_dim)
    """
    model.eval()

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    if context_mode == "isolated":
        base_ids = []
        token_pos = 1
    elif context_mode == "template":
        base_ids = tokenizer(template_prefix, add_special_tokens=False).input_ids
        token_pos = 1 + len(base_ids)
    else:
        raise ValueError("context_mode must be 'isolated' or 'template'")

    all_vecs = {L: [] for L in layers}

    for start in range(0, len(token_ids), batch_size):
        batch_ids = [int(t) for t in token_ids[start:start + batch_size]]

        input_ids_list = []
        for tok_id in batch_ids:
            full_ids = [cls_id] + base_ids + [tok_id] + [sep_id]
            input_ids_list.append(full_ids)

        max_len = max(len(x) for x in input_ids_list)

        padded_input_ids = []
        attention_masks = []
        token_type_ids = []

        for ids in input_ids_list:
            pad_len = max_len - len(ids)
            padded_input_ids.append(ids + [pad_id] * pad_len)
            attention_masks.append([1] * len(ids) + [0] * pad_len)
            token_type_ids.append([0] * max_len)

        input_ids = torch.tensor(padded_input_ids, dtype=torch.long, device=device)
        attention_mask = torch.tensor(attention_masks, dtype=torch.long, device=device)
        token_type_ids = torch.tensor(token_type_ids, dtype=torch.long, device=device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,
            return_dict=True,
        )

        for L in layers:
            vecs = outputs.hidden_states[L][:, token_pos, :].detach().cpu().numpy()
            all_vecs[L].append(vecs)

    return {L: np.vstack(all_vecs[L]) for L in layers}

In [ ]:
layers = (5, 8, 11, 12)

# extract isolated embeddings
pt_iso = extract_token_hidden_states(
    model=pt_bert,
    tokenizer=tokenizer,
    token_ids=unique_token_ids,
    device=device,
    layers=layers,
    batch_size=64,
    context_mode="isolated",
)

ft_iso = extract_token_hidden_states(
    model=ft_bert,
    tokenizer=tokenizer,
    token_ids=unique_token_ids,
    device=device,
    layers=layers,
    batch_size=64,
    context_mode="isolated",
)

# extract templated embeddings
pt_tpl = extract_token_hidden_states(
    model=pt_bert,
    tokenizer=tokenizer,
    token_ids=unique_token_ids,
    device=device,
    layers=layers,
    batch_size=64,
    context_mode="template",
)

ft_tpl = extract_token_hidden_states(
    model=ft_bert,
    tokenizer=tokenizer,
    token_ids=unique_token_ids,
    device=device,
    layers=layers,
    batch_size=64,
    context_mode="template",
)

# store as vectors in the dataframe
for L in layers:
    token_df[f"pt_iso_l{L}"] = list(pt_iso[L])
    token_df[f"ft_iso_l{L}"] = list(ft_iso[L])
    token_df[f"pt_tpl_l{L}"] = list(pt_tpl[L])
    token_df[f"ft_tpl_l{L}"] = list(ft_tpl[L])

token_df.head()

# Find the nearest neighbour of an itemset

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

def stack_embs(series):
    return np.vstack(series.apply(np.asarray).to_list())

def add_nearest_neighbors(
    query_df,
    token_df,
    query_emb_col,
    token_emb_col,
    prefix,
    k=1,
    metric="cosine",   # "cosine" or "euclidean"
    exclude_constituents=False,
    drop_embedding_cols=False,
    embedding_cols_to_drop=None,
):
    q_mat = stack_embs(query_df[query_emb_col])
    t_mat = stack_embs(token_df[token_emb_col])

    if metric == "cosine":
        scores = cosine_similarity(q_mat, t_mat)
        larger_is_better = True
        score_col = f"{prefix}_nn_cosines"
    elif metric == "euclidean":
        scores = euclidean_distances(q_mat, t_mat)
        larger_is_better = False
        score_col = f"{prefix}_nn_euclidean"
    else:
        raise ValueError("metric must be 'cosine' or 'euclidean'")

    token_ids = token_df["token_id"].values
    token_strs = token_df["token"].values

    nn_token_ids = []
    nn_tokens = []
    nn_scores = []

    for i in range(len(query_df)):
        row = scores[i].copy()

        if exclude_constituents:
            itemset_ids = query_df.iloc[i]["ordered_ids"]
            if larger_is_better:
                row[np.isin(token_ids, list(itemset_ids))] = -np.inf
            else:
                row[np.isin(token_ids, list(itemset_ids))] = np.inf

        top_idx = np.argsort(row)[::-1][:k] if larger_is_better else np.argsort(row)[:k]

        nn_token_ids.append(token_ids[top_idx].tolist())
        nn_tokens.append(token_strs[top_idx].tolist())
        nn_scores.append(row[top_idx].tolist())

    out = query_df.copy()
    out[f"{prefix}_nn_token_ids"] = nn_token_ids
    out[f"{prefix}_nn_tokens"] = nn_tokens
    out[score_col] = nn_scores
    out[f"{prefix}_nn_metric"] = metric

    if drop_embedding_cols:
        if embedding_cols_to_drop is None:
            embedding_cols_to_drop = [c for c in out.columns if "_l" in c]
        cols_to_drop = [c for c in embedding_cols_to_drop if c in out.columns]
        out = out.drop(columns=cols_to_drop)

    return out

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

def add_euclidean_distances(
    df_with_nns,
    query_df,
    token_df,
    query_emb_col,
    token_emb_col,
    nn_ids_col,
    out_col,
):
    out = df_with_nns.copy()

    q_lookup = {
        tuple(row["ordered_ids"]): np.asarray(row[query_emb_col])
        for _, row in query_df.iterrows()
    }
    t_lookup = {
        row["token_id"]: np.asarray(row[token_emb_col])
        for _, row in token_df.iterrows()
    }

    all_dists = []

    for _, row in out.iterrows():
        q_vec = q_lookup[tuple(row["ordered_ids"])].reshape(1, -1)
        dists = []

        for tok_id in row[nn_ids_col]:
            t_vec = t_lookup[tok_id].reshape(1, -1)
            dists.append(euclidean_distances(q_vec, t_vec)[0, 0])

        all_dists.append(dists)

    out[out_col] = all_dists
    return out

def add_cosine_values(
    df_with_nns,
    query_df,
    token_df,
    query_emb_col,
    token_emb_col,
    nn_ids_col,
    out_col,
    as_distance=False,
):
    out = df_with_nns.copy()

    q_lookup = {
        tuple(row["ordered_ids"]): np.asarray(row[query_emb_col])
        for _, row in query_df.iterrows()
    }
    t_lookup = {
        row["token_id"]: np.asarray(row[token_emb_col])
        for _, row in token_df.iterrows()
    }

    all_vals = []

    for _, row in out.iterrows():
        q_vec = q_lookup[tuple(row["ordered_ids"])].reshape(1, -1)
        vals = []

        for tok_id in row[nn_ids_col]:
            t_vec = t_lookup[tok_id].reshape(1, -1)
            cos_sim = cosine_similarity(q_vec, t_vec)[0, 0]
            vals.append(1 - cos_sim if as_distance else cos_sim)

        all_vals.append(vals)

    out[out_col] = all_vals
    return out

In [ ]:
import numpy as np

def add_distance_deltas(
    df,
    ft_euclidean_col,
    pt_euclidean_col,
    ft_cosine_col,
    pt_cosine_col,
    out_ft_cosine_distance_col,
    out_delta_euclidean_col,
    out_delta_cosine_col,
    out_mean_delta_euclidean_col=None,
    out_mean_delta_cosine_col=None,
    ft_cosine_is_similarity=True,
):
    out = df.copy()

    if ft_cosine_is_similarity:
        out[out_ft_cosine_distance_col] = out[ft_cosine_col].apply(
            lambda xs: [1 - x for x in xs]
        )
        ft_cosine_distance_col = out_ft_cosine_distance_col
    else:
        ft_cosine_distance_col = ft_cosine_col

    out[out_delta_euclidean_col] = [
        np.abs(np.array(pt) - np.array(ft)).tolist()
        for ft, pt in zip(out[ft_euclidean_col], out[pt_euclidean_col])
    ]

    out[out_delta_cosine_col] = [
        np.abs(np.array(pt) - np.array(ft)).tolist()
        for ft, pt in zip(out[ft_cosine_distance_col], out[pt_cosine_col])
    ]

    if out_mean_delta_euclidean_col is not None:
        out[out_mean_delta_euclidean_col] = out[out_delta_euclidean_col].apply(np.mean)

    if out_mean_delta_cosine_col is not None:
        out[out_mean_delta_cosine_col] = out[out_delta_cosine_col].apply(np.mean)

    return out

In [ ]:
def plot_rating_proxy_vs_non_rating_proxy(
    df,
    anchor_layer,
    context="iso",
):
    """
    Compare explicit Amazon rating proxies with all other candidates.
    """
    assert context in {"iso", "tpl"}

    df_plot = add_amazon_candidate_group(df)

    df_rating = df_plot[
        df_plot["candidate_group"] == "rating_proxy"
    ]
    df_non_rating = df_plot[
        df_plot["candidate_group"] == "non_rating_proxy"
    ]

    print(
        f"Layer {anchor_layer}, {context}: "
        f"{len(df_rating)} rating proxies and "
        f"{len(df_non_rating)} non-rating proxies"
    )

    if context == "iso":
        euc_name = (
            lambda layer:
            f"mean_delta_l{anchor_layer}_euclidean_l{layer}"
        )
        cos_name = (
            lambda layer:
            f"mean_delta_l{anchor_layer}_cosine_l{layer}"
        )
        title_prefix = (
            f"Layer-{anchor_layer} anchor, isolated"
        )
    else:
        euc_name = (
            lambda layer:
            f"mean_delta_tpl_l{anchor_layer}_euclidean_l{layer}"
        )
        cos_name = (
            lambda layer:
            f"mean_delta_tpl_l{anchor_layer}_cosine_l{layer}"
        )
        title_prefix = (
            f"Layer-{anchor_layer} anchor, templated"
        )

    summary_rows = []

    for layer in LAYERS:
        for group_name, group_df in (
            ("rating_proxy", df_rating),
            ("non_rating_proxy", df_non_rating),
        ):
            summary_rows.append({
                "group": group_name,
                "count": len(group_df),
                "layer": f"L{layer}",
                "mean_delta_euclidean": (
                    group_df[euc_name(layer)].mean()
                ),
                "std_delta_euclidean": (
                    group_df[euc_name(layer)].std()
                ),
                "mean_delta_cosine": (
                    group_df[cos_name(layer)].mean()
                ),
                "std_delta_cosine": (
                    group_df[cos_name(layer)].std()
                ),
            })

    summary = pd.DataFrame(summary_rows)

    x = np.arange(len(LAYERS))
    width = 0.35

    rating = summary[
        summary["group"] == "rating_proxy"
    ]
    non_rating = summary[
        summary["group"] == "non_rating_proxy"
    ]

    rating_euc_std = np.nan_to_num(
        rating["std_delta_euclidean"].to_numpy(),
        nan=0.0,
    )
    non_rating_euc_std = np.nan_to_num(
        non_rating["std_delta_euclidean"].to_numpy(),
        nan=0.0,
    )
    rating_cos_std = np.nan_to_num(
        rating["std_delta_cosine"].to_numpy(),
        nan=0.0,
    )
    non_rating_cos_std = np.nan_to_num(
        non_rating["std_delta_cosine"].to_numpy(),
        nan=0.0,
    )

    # Euclidean comparison
    plt.figure(figsize=(8, 5))

    plt.bar(
        x - width / 2,
        rating["mean_delta_euclidean"].to_numpy(),
        width,
        yerr=rating_euc_std,
        label="explicit rating proxy",
        capsize=4,
    )
    plt.bar(
        x + width / 2,
        non_rating["mean_delta_euclidean"].to_numpy(),
        width,
        yerr=non_rating_euc_std,
        label="non-rating proxy",
        capsize=4,
    )

    plt.xticks(x, [f"L{layer}" for layer in LAYERS])
    plt.ylabel("average mean delta euclidean")
    plt.title(
        f"{title_prefix}: rating proxy vs non-rating proxy"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Cosine comparison
    plt.figure(figsize=(8, 5))

    plt.bar(
        x - width / 2,
        rating["mean_delta_cosine"].to_numpy(),
        width,
        yerr=rating_cos_std,
        label="explicit rating proxy",
        capsize=4,
    )
    plt.bar(
        x + width / 2,
        non_rating["mean_delta_cosine"].to_numpy(),
        width,
        yerr=non_rating_cos_std,
        label="non-rating proxy",
        capsize=4,
    )

    plt.xticks(x, [f"L{layer}" for layer in LAYERS])
    plt.ylabel("average mean delta cosine")
    plt.title(
        f"{title_prefix}: rating proxy vs non-rating proxy"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()

    return summary


In [ ]:
#### Version with Standard dev
def plot_rating_proxy_vs_non_rating_proxy(
    df,
    anchor_layer,
    context="iso",
):
    assert context in {"iso", "tpl"}

    pattern = re.compile(r'^[0-9\-]+$')

    def tokens_to_string(x):
        return " ".join(x) if isinstance(x, list) else str(x)

    df_plot = df.copy()
    df_plot["token_str"] = df_plot["ordered_tokens"].apply(tokens_to_string)

    is_numeric_dash = df_plot["token_str"].apply(
        lambda x: bool(pattern.fullmatch(x.replace(" ", "")))
    )

    df_numeric_dash = df_plot[is_numeric_dash]
    df_non_numeric = df_plot[~is_numeric_dash]

    if context == "iso":
        euc_name = lambda L: f"mean_delta_l{anchor_layer}_euclidean_l{L}"
        cos_name = lambda L: f"mean_delta_l{anchor_layer}_cosine_l{L}"
        title_prefix = f"Layer-{anchor_layer} anchor, isolated"
    else:
        euc_name = lambda L: f"mean_delta_tpl_l{anchor_layer}_euclidean_l{L}"
        cos_name = lambda L: f"mean_delta_tpl_l{anchor_layer}_cosine_l{L}"
        title_prefix = f"Layer-{anchor_layer} anchor, templated"

    summary_rows = []
    for L in LAYERS:
        summary_rows.append({
            "group": "numeric",
            "layer": f"L{L}",
            "mean_delta_euclidean": df_numeric_dash[euc_name(L)].mean(),
            "std_delta_euclidean": df_numeric_dash[euc_name(L)].std(),
            "mean_delta_cosine": df_numeric_dash[cos_name(L)].mean(),
            "std_delta_cosine": df_numeric_dash[cos_name(L)].std(),
        })
        summary_rows.append({
            "group": "non_numeric",
            "layer": f"L{L}",
            "mean_delta_euclidean": df_non_numeric[euc_name(L)].mean(),
            "std_delta_euclidean": df_non_numeric[euc_name(L)].std(),
            "mean_delta_cosine": df_non_numeric[cos_name(L)].mean(),
            "std_delta_cosine": df_non_numeric[cos_name(L)].std(),
        })

    summary = pd.DataFrame(summary_rows)

    x = np.arange(len(LAYERS))
    width = 0.35

    # --- Euclidean ---
    plt.figure(figsize=(8, 5))

    num = summary[summary["group"] == "numeric"]
    non = summary[summary["group"] == "non_numeric"]

    plt.bar(
        x - width/2,
        num["mean_delta_euclidean"].values,
        width,
        yerr=num["std_delta_euclidean"].values,
        label="numeric",
        capsize=4
    )

    plt.bar(
        x + width/2,
        non["mean_delta_euclidean"].values,
        width,
        yerr=non["std_delta_euclidean"].values,
        label="non_numeric",
        capsize=4
    )

    plt.xticks(x, [f"L{L}" for L in LAYERS])
    plt.ylabel("average mean delta euclidean")
    plt.title(f"{title_prefix}: numeric vs non-numeric")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- Cosine ---
    plt.figure(figsize=(8, 5))

    plt.bar(
        x - width/2,
        num["mean_delta_cosine"].values,
        width,
        yerr=num["std_delta_cosine"].values,
        label="numeric",
        capsize=4
    )

    plt.bar(
        x + width/2,
        non["mean_delta_cosine"].values,
        width,
        yerr=non["std_delta_cosine"].values,
        label="non_numeric",
        capsize=4
    )

    plt.xticks(x, [f"L{L}" for L in LAYERS])
    plt.ylabel("average mean delta cosine")
    plt.title(f"{title_prefix}: numeric vs non-numeric")
    plt.legend()
    plt.tight_layout()
    plt.show()


#### Version with Standard dev

### Anchor layer 12

### Top 10 Nearest Neighbours

Calculated in finetuned embedding space

Isolated

In [ ]:
df_iso_l12 = run_anchor_analysis(df_mean, token_df, anchor_layer=12, context="iso")
plot_anchor_overlay(df_iso_l12, anchor_layer=12, context="iso")
plot_rating_proxy_vs_non_rating_proxy(df_iso_l12, anchor_layer=12, context="iso")

Templated

In [ ]:
df_tpl_l12 = run_anchor_analysis(df_mean, token_df, anchor_layer=12, context="tpl")
plot_anchor_overlay(df_tpl_l12, anchor_layer=12, context="tpl")
plot_rating_proxy_vs_non_rating_proxy(df_tpl_l12, anchor_layer=12, context="tpl")

### Anchor layer 11

Isolated

In [ ]:
df_iso_l11 = run_anchor_analysis(df_mean, token_df, anchor_layer=11, context="iso")
plot_anchor_overlay(df_iso_l11, anchor_layer=11, context="iso")
plot_rating_proxy_vs_non_rating_proxy(df_iso_l11, anchor_layer=11, context="iso")

Templated


In [ ]:
df_tpl_l11 = run_anchor_analysis(df_mean, token_df, anchor_layer=11, context="tpl")
plot_anchor_overlay(df_tpl_l11, anchor_layer=11, context="tpl")
plot_rating_proxy_vs_non_rating_proxy(df_tpl_l11, anchor_layer=11, context="tpl")

### Anchor Layer 8

Isolated

In [ ]:
df_iso_l8 = run_anchor_analysis(df_mean, token_df, anchor_layer=8, context="iso")
plot_anchor_overlay(df_iso_l8, anchor_layer=8, context="iso")
plot_rating_proxy_vs_non_rating_proxy(df_iso_l8, anchor_layer=8, context="iso")

Templated

In [ ]:
df_tpl_l8 = run_anchor_analysis(df_mean, token_df, anchor_layer=8, context="tpl")
plot_anchor_overlay(df_tpl_l8, anchor_layer=8, context="tpl")
plot_rating_proxy_vs_non_rating_proxy(df_tpl_l8, anchor_layer=8, context="tpl")

### Anchor Layer 5

Isolate 

In [ ]:
df_iso_l5 = run_anchor_analysis(df_mean, token_df, anchor_layer=5, context="iso")
plot_anchor_overlay(df_iso_l5, anchor_layer=5, context="iso")
plot_rating_proxy_vs_non_rating_proxy(df_iso_l5, anchor_layer=5, context="iso")

Templated

In [ ]:
df_tpl_l5 = run_anchor_analysis(df_mean, token_df, anchor_layer=5, context="tpl")
plot_anchor_overlay(df_tpl_l5, anchor_layer=5, context="tpl")
plot_rating_proxy_vs_non_rating_proxy(df_tpl_l5, anchor_layer=5, context="tpl")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_anchor_same_layer_with_std(
    df,
    anchor_layer,
    context="iso",   # "iso" or "tpl"
    top_n=50,
    min_length=2,
    sort_col="support",
):
    assert context in {"iso", "tpl"}

    if context == "iso":
        euc_mean_col = f"mean_delta_l{anchor_layer}_euclidean_l{anchor_layer}"
        euc_raw_col  = f"delta_l{anchor_layer}_euclidean_l{anchor_layer}"
        cos_mean_col = f"mean_delta_l{anchor_layer}_cosine_l{anchor_layer}"
        cos_raw_col  = f"delta_l{anchor_layer}_cosine_l{anchor_layer}"
        title_prefix = f"Layer-{anchor_layer} anchor, isolated"
    else:
        euc_mean_col = f"mean_delta_tpl_l{anchor_layer}_euclidean_l{anchor_layer}"
        euc_raw_col  = f"delta_tpl_l{anchor_layer}_euclidean_l{anchor_layer}"
        cos_mean_col = f"mean_delta_tpl_l{anchor_layer}_cosine_l{anchor_layer}"
        cos_raw_col  = f"delta_tpl_l{anchor_layer}_cosine_l{anchor_layer}"
        title_prefix = f"Layer-{anchor_layer} anchor, templated"

    plot_df = (
        df[df["length"] >= min_length]
        .sort_values(sort_col, ascending=False)
        .head(top_n)
        .copy()
    )

    plot_df["label"] = plot_df["ordered_tokens"].apply(
        lambda x: " ".join(x) if isinstance(x, list) else str(x)
    )

    # std over the raw NN delta values for each itemset
    plot_df["std_euclidean"] = plot_df[euc_raw_col].apply(
        lambda xs: float(np.std(xs)) if len(xs) > 0 else 0.0
    )
    plot_df["std_cosine"] = plot_df[cos_raw_col].apply(
        lambda xs: float(np.std(xs)) if len(xs) > 0 else 0.0
    )

    x = np.arange(len(plot_df))

    # Euclidean
    plt.figure(figsize=(20, 6))
    plt.bar(
        x,
        plot_df[euc_mean_col],
        yerr=plot_df["std_euclidean"],
        capsize=4
    )
    plt.xticks(x, plot_df["label"], rotation=75, ha="right")
    plt.ylabel("mean delta euclidean")
    plt.title(f"{title_prefix}: same-layer delta euclidean only")
    plt.tight_layout()
    plt.show()

    # Cosine
    plt.figure(figsize=(20, 6))
    plt.bar(
        x,
        plot_df[cos_mean_col],
        yerr=plot_df["std_cosine"],
        capsize=4
    )
    plt.xticks(x, plot_df["label"], rotation=75, ha="right")
    plt.ylabel("mean delta cosine")
    plt.title(f"{title_prefix}: same-layer delta cosine only")
    plt.tight_layout()
    plt.show()

    return plot_df[
        [
            "label",
            euc_mean_col,
            "std_euclidean",
            cos_mean_col,
            "std_cosine",
        ]
    ]

In [ ]:
plot_anchor_same_layer_with_std(df_iso_l12, anchor_layer=12, context="iso")
plot_anchor_same_layer_with_std(df_iso_l11, anchor_layer=11, context="iso")
plot_anchor_same_layer_with_std(df_iso_l8,  anchor_layer=8,  context="iso")
plot_anchor_same_layer_with_std(df_iso_l5,  anchor_layer=5,  context="iso")

In [ ]:
plot_anchor_same_layer_with_std(df_tpl_l12, anchor_layer=12, context="tpl")
plot_anchor_same_layer_with_std(df_tpl_l11, anchor_layer=11, context="tpl")
plot_anchor_same_layer_with_std(df_tpl_l8,  anchor_layer=8,  context="tpl")
plot_anchor_same_layer_with_std(df_tpl_l5,  anchor_layer=5,  context="tpl")

In [ ]:
ANCHOR_LAYERS = (12, 11, 8, 5)


def build_matched_anchor_summary(
    anchor_dfs,
    context="iso",
):
    """
    Build the matched anchor/layer comparison for explicit Amazon
    rating proxies versus all non-rating-proxy candidates.
    """
    assert context in {"iso", "tpl"}

    rows = []

    for anchor_layer in ANCHOR_LAYERS:
        df_plot = add_amazon_candidate_group(
            anchor_dfs[anchor_layer]
        )

        df_rating = df_plot[
            df_plot["candidate_group"] == "rating_proxy"
        ]
        df_non_rating = df_plot[
            df_plot["candidate_group"] == "non_rating_proxy"
        ]

        if context == "iso":
            euc_col = (
                f"mean_delta_l{anchor_layer}_"
                f"euclidean_l{anchor_layer}"
            )
            cos_col = (
                f"mean_delta_l{anchor_layer}_"
                f"cosine_l{anchor_layer}"
            )
        else:
            euc_col = (
                f"mean_delta_tpl_l{anchor_layer}_"
                f"euclidean_l{anchor_layer}"
            )
            cos_col = (
                f"mean_delta_tpl_l{anchor_layer}_"
                f"cosine_l{anchor_layer}"
            )

        for group_name, group_df in (
            ("rating_proxy", df_rating),
            ("non_rating_proxy", df_non_rating),
        ):
            rows.append({
                "group": group_name,
                "count": len(group_df),
                "layer": f"L{anchor_layer}",
                "mean_delta_euclidean": (
                    group_df[euc_col].mean()
                ),
                "std_delta_euclidean": (
                    group_df[euc_col].std()
                ),
                "mean_delta_cosine": (
                    group_df[cos_col].mean()
                ),
                "std_delta_cosine": (
                    group_df[cos_col].std()
                ),
            })

    summary = pd.DataFrame(rows)

    layer_order = [
        f"L{layer}"
        for layer in ANCHOR_LAYERS
    ]

    summary["layer"] = pd.Categorical(
        summary["layer"],
        categories=layer_order,
        ordered=True,
    )

    return (
        summary
        .sort_values(["layer", "group"])
        .reset_index(drop=True)
    )


def plot_matched_anchor_deltas(
    anchor_dfs,
    context="iso",
):
    """
    Plot matched anchor/layer deltas with standard-deviation bars.
    """
    summary = build_matched_anchor_summary(
        anchor_dfs,
        context=context,
    )

    layer_order = [
        f"L{layer}"
        for layer in ANCHOR_LAYERS
    ]

    rating = (
        summary[
            summary["group"] == "rating_proxy"
        ]
        .set_index("layer")
        .loc[layer_order]
    )
    non_rating = (
        summary[
            summary["group"] == "non_rating_proxy"
        ]
        .set_index("layer")
        .loc[layer_order]
    )

    x = np.arange(len(layer_order))
    width = 0.35

    title_prefix = (
        "Matched anchor/layer only, isolated"
        if context == "iso"
        else "Matched anchor/layer only, templated"
    )

    rating_euc_std = np.nan_to_num(
        rating["std_delta_euclidean"].to_numpy(),
        nan=0.0,
    )
    non_rating_euc_std = np.nan_to_num(
        non_rating["std_delta_euclidean"].to_numpy(),
        nan=0.0,
    )
    rating_cos_std = np.nan_to_num(
        rating["std_delta_cosine"].to_numpy(),
        nan=0.0,
    )
    non_rating_cos_std = np.nan_to_num(
        non_rating["std_delta_cosine"].to_numpy(),
        nan=0.0,
    )

    # Euclidean
    plt.figure(figsize=(8, 5))
    plt.bar(
        x - width / 2,
        rating["mean_delta_euclidean"].to_numpy(),
        width,
        yerr=rating_euc_std,
        label="explicit rating proxy",
        capsize=4,
    )
    plt.bar(
        x + width / 2,
        non_rating["mean_delta_euclidean"].to_numpy(),
        width,
        yerr=non_rating_euc_std,
        label="non-rating proxy",
        capsize=4,
    )
    plt.xticks(x, layer_order)
    plt.ylabel("average mean delta euclidean")
    plt.title(
        f"{title_prefix}: rating proxy vs non-rating proxy"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Cosine
    plt.figure(figsize=(8, 5))
    plt.bar(
        x - width / 2,
        rating["mean_delta_cosine"].to_numpy(),
        width,
        yerr=rating_cos_std,
        label="explicit rating proxy",
        capsize=4,
    )
    plt.bar(
        x + width / 2,
        non_rating["mean_delta_cosine"].to_numpy(),
        width,
        yerr=non_rating_cos_std,
        label="non-rating proxy",
        capsize=4,
    )
    plt.xticks(x, layer_order)
    plt.ylabel("average mean delta cosine")
    plt.title(
        f"{title_prefix}: rating proxy vs non-rating proxy"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()

    return summary


In [ ]:
iso_anchor_dfs = {
    12: df_iso_l12,
    11: df_iso_l11,
     8: df_iso_l8,
     5: df_iso_l5,
}

summary_iso_matched = plot_matched_anchor_deltas(iso_anchor_dfs, context="iso")
summary_iso_matched

In [ ]:
tpl_anchor_dfs = {
    12: df_tpl_l12,
    11: df_tpl_l11,
     8: df_tpl_l8,
     5: df_tpl_l5,
}

summary_tpl_matched = plot_matched_anchor_deltas(tpl_anchor_dfs, context="tpl")
summary_tpl_matched

DF_First

In [ ]:
df_tpl_first_l12 = run_anchor_analysis(df_first_token, token_df, anchor_layer=12, context="tpl")
df_tpl_first_l11 = run_anchor_analysis(df_first_token, token_df, anchor_layer=11, context="tpl")
df_tpl_first_l8  = run_anchor_analysis(df_first_token, token_df, anchor_layer=8,  context="tpl")
df_tpl_first_l5  = run_anchor_analysis(df_first_token, token_df, anchor_layer=5,  context="tpl")

tpl_first_anchor_dfs = {
    12: df_tpl_first_l12,
    11: df_tpl_first_l11,
     8: df_tpl_first_l8,
     5: df_tpl_first_l5,
}

summary_tpl_first_matched = plot_matched_anchor_deltas(tpl_first_anchor_dfs, context="tpl")
summary_tpl_first_matched

DF_Sum

In [ ]:
df_tpl_sum_l12 = run_anchor_analysis(df_sum, token_df, anchor_layer=12, context="tpl")
df_tpl_sum_l11 = run_anchor_analysis(df_sum, token_df, anchor_layer=11, context="tpl")
df_tpl_sum_l8  = run_anchor_analysis(df_sum, token_df, anchor_layer=8,  context="tpl")
df_tpl_sum_l5  = run_anchor_analysis(df_sum, token_df, anchor_layer=5,  context="tpl")

tpl_sum_anchor_dfs = {
    12: df_tpl_sum_l12,
    11: df_tpl_sum_l11,
     8: df_tpl_sum_l8,
     5: df_tpl_sum_l5,
}

summary_tpl_sum_matched = plot_matched_anchor_deltas(tpl_sum_anchor_dfs, context="tpl")
summary_tpl_sum_matched

---
## New method: genuinely contiguous salient n-grams

The original FP-growth method treats the top salient tokens in each review as
an unordered transaction. Sorting an itemset afterward gives it an order, but
does not prove that its tokens were adjacent.

This method instead uses the original BERT token positions stored in
`topk_vocab`. It retains a sequence only when:

1. every constituent token belongs to the top-*k* salient tokens;
2. the token positions are consecutive in the model input;
3. the same ordered token sequence reaches the minimum document support.

Support is document-level: an n-gram contributes at most once per sampled
review.


In [ ]:
def extract_contiguous_salient_ngrams(
    salient_token_info,
    tokenizer,
    ngram_lengths=(2, 3),
    min_count=5,
):
    """
    Extract genuinely contiguous n-grams from the salient-token positions.

    Parameters
    ----------
    salient_token_info:
        Output created by build_salience_vocab_matrix_from_indices().
    tokenizer:
        The tokenizer used by the fine-tuned model.
    ngram_lengths:
        Contiguous sequence lengths to retain.
    min_count:
        Minimum number of sampled reviews containing the n-gram.

    Returns
    -------
    pandas.DataFrame
        One row per ordered contiguous n-gram.
    """
    number_of_documents = len(salient_token_info)

    document_counts = Counter()
    document_saliency_values = defaultdict(list)

    for example in salient_token_info:
        topk = sorted(
            example["topk_vocab"],
            key=lambda item: int(item["position"]),
        )

        # Store at most one saliency value per n-gram per document.
        ngram_saliency_in_document = {}

        for n in ngram_lengths:
            if len(topk) < n:
                continue

            for start in range(len(topk) - n + 1):
                window = topk[start:start + n]

                positions = [
                    int(item["position"])
                    for item in window
                ]

                is_contiguous = all(
                    right - left == 1
                    for left, right in zip(
                        positions,
                        positions[1:],
                    )
                )

                if not is_contiguous:
                    continue

                ordered_ids = tuple(
                    int(item["token_id"])
                    for item in window
                )

                mean_saliency = float(np.mean([
                    float(item["saliency_norm"])
                    for item in window
                ]))

                # If the same n-gram occurs more than once in one review,
                # retain its strongest occurrence but count the review once.
                previous = ngram_saliency_in_document.get(
                    ordered_ids,
                    -np.inf,
                )
                ngram_saliency_in_document[ordered_ids] = max(
                    previous,
                    mean_saliency,
                )

        for ordered_ids, mean_saliency in ngram_saliency_in_document.items():
            document_counts[ordered_ids] += 1
            document_saliency_values[ordered_ids].append(mean_saliency)

    rows = []

    for ordered_ids, count in document_counts.items():
        if count < min_count:
            continue

        ordered_tokens = tokenizer.convert_ids_to_tokens(
            list(ordered_ids)
        )
        support = count / number_of_documents
        saliency_score = float(np.mean(
            document_saliency_values[ordered_ids]
        ))

        rows.append({
            "ordered_ids": list(ordered_ids),
            "ordered_tokens": ordered_tokens,
            "ngram_text": tokenizer.convert_tokens_to_string(
                ordered_tokens
            ),
            "length": len(ordered_ids),
            "count": count,
            "support": support,
            "saliency_score": saliency_score,
            "score": support * saliency_score,
        })

    result = pd.DataFrame(rows)

    if result.empty:
        return pd.DataFrame(columns=[
            "ordered_ids",
            "ordered_tokens",
            "ngram_text",
            "length",
            "count",
            "support",
            "saliency_score",
            "score",
        ])

    return result.sort_values(
        ["score", "support"],
        ascending=False,
    ).reset_index(drop=True)


In [ ]:
MIN_CONTIGUOUS_COUNT = 5
NGRAM_LENGTHS = (2, 3)

actual_ngrams_neg_5 = extract_contiguous_salient_ngrams(
    salient_token_info_neg,
    tokenizer,
    ngram_lengths=NGRAM_LENGTHS,
    min_count=MIN_CONTIGUOUS_COUNT,
)

actual_ngrams_neg_10 = extract_contiguous_salient_ngrams(
    salient_token_info_neg_10,
    tokenizer,
    ngram_lengths=NGRAM_LENGTHS,
    min_count=MIN_CONTIGUOUS_COUNT,
)

actual_ngrams_neg_5 = add_amazon_candidate_group(
    actual_ngrams_neg_5
)
actual_ngrams_neg_10 = add_amazon_candidate_group(
    actual_ngrams_neg_10
)

print("Top-5 salient-token setup:")
print("  actual contiguous n-grams:", len(actual_ngrams_neg_5))

print("Top-10 salient-token setup:")
print("  actual contiguous n-grams:", len(actual_ngrams_neg_10))


### Explicit Amazon rating proxies among all actual n-grams

This table shows every contiguous n-gram that meets the support threshold and
is classified as an explicit rating-value proxy. These form the proposed
shortcut-candidate group; the remaining n-grams form the comparison group.


In [ ]:
print(
    actual_ngrams_neg_10["candidate_group"]
    .value_counts()
)

actual_rating_proxy_ngrams = (
    actual_ngrams_neg_10[
        actual_ngrams_neg_10["is_rating_proxy"]
    ]
    .sort_values(
        ["score", "support"],
        ascending=False,
    )
    .reset_index(drop=True)
)

actual_rating_proxy_ngrams[
    [
        "candidate_text",
        "ordered_tokens",
        "length",
        "count",
        "support",
        "saliency_score",
        "score",
    ]
]


### Actual contiguous bigrams

In [ ]:
actual_ngrams_neg_10[
    actual_ngrams_neg_10["length"] == 2
].sort_values(
    ["support", "saliency_score"],
    ascending=False,
).head(40)


### Actual contiguous trigrams

In [ ]:
actual_ngrams_neg_10[
    actual_ngrams_neg_10["length"] == 3
].sort_values(
    ["support", "saliency_score"],
    ascending=False,
).head(40)


### Candidate table for subsequent experiments

`actual_ngram_candidates` can be passed to the same hidden-state extraction
functions used in Experiment 1 because it contains `ordered_ids`,
`ordered_tokens`, `support`, `saliency_score`, and `score`.


In [ ]:
actual_ngram_candidates = (
    actual_ngrams_neg_10
    .sort_values("score", ascending=False)
    .head(100)
    .copy()
)

# The classifier has already been applied to the complete n-gram table,
# but applying it again makes this cell independently robust.
actual_ngram_candidates = add_amazon_candidate_group(
    actual_ngram_candidates
)

print(
    actual_ngram_candidates["candidate_group"]
    .value_counts()
)

display(
    actual_ngram_candidates[
        actual_ngram_candidates["is_rating_proxy"]
    ][
        [
            "candidate_text",
            "ordered_tokens",
            "length",
            "count",
            "support",
            "saliency_score",
            "score",
        ]
    ]
    .sort_values("score", ascending=False)
    .head(50)
)

actual_ngram_candidates.head(20)


### Representation-shift experiment on actual n-grams

In [ ]:
actual_ngram_ids = actual_ngram_candidates["ordered_ids"].tolist()
layers = (5, 8, 11, 12)

# Pretrained BERT
actual_pt_iso, actual_pt_logits_iso = extract_itemset_hidden_states(
    pt_bert,
    tokenizer,
    actual_ngram_ids,
    device,
    layers=layers,
    context_mode="isolated",
)

actual_pt_tpl, actual_pt_logits_tpl = extract_itemset_hidden_states(
    pt_bert,
    tokenizer,
    actual_ngram_ids,
    device,
    layers=layers,
    context_mode="template",
)

# Fine-tuned Amazon Polarity BERT
actual_ft_iso, actual_ft_logits_iso = extract_itemset_hidden_states(
    ft_bert,
    tokenizer,
    actual_ngram_ids,
    device,
    layers=layers,
    context_mode="isolated",
)

actual_ft_tpl, actual_ft_logits_tpl = extract_itemset_hidden_states(
    ft_bert,
    tokenizer,
    actual_ngram_ids,
    device,
    layers=layers,
    context_mode="template",
)

for layer in layers:
    actual_ngram_candidates[f"pt_iso_l{layer}"] = list(
        actual_pt_iso[layer]
    )
    actual_ngram_candidates[f"pt_tpl_l{layer}"] = list(
        actual_pt_tpl[layer]
    )
    actual_ngram_candidates[f"ft_iso_l{layer}"] = list(
        actual_ft_iso[layer]
    )
    actual_ngram_candidates[f"ft_tpl_l{layer}"] = list(
        actual_ft_tpl[layer]
    )

actual_ngram_candidates["ft_iso_logits"] = list(
    actual_ft_logits_iso
)
actual_ngram_candidates["ft_tpl_logits"] = list(
    actual_ft_logits_tpl
)

actual_ngram_candidates.head()


In [ ]:
actual_shift_table = actual_ngram_candidates[
    [
        "ordered_ids",
        "ordered_tokens",
        "ngram_text",
        "length",
        "count",
        "support",
        "saliency_score",
        "score",
        "candidate_text",
        "is_rating_proxy",
        "candidate_group",
    ]
].copy()

for layer in (5, 8, 11, 12):
    actual_shift_table[f"cos_shift_iso_l{layer}"] = [
        cos_shift(pt, ft)
        for pt, ft in zip(
            actual_ngram_candidates[f"pt_iso_l{layer}"],
            actual_ngram_candidates[f"ft_iso_l{layer}"],
        )
    ]

    actual_shift_table[f"euc_iso_l{layer}"] = [
        euc_dist(pt, ft)
        for pt, ft in zip(
            actual_ngram_candidates[f"pt_iso_l{layer}"],
            actual_ngram_candidates[f"ft_iso_l{layer}"],
        )
    ]

    actual_shift_table[f"cos_shift_tpl_l{layer}"] = [
        cos_shift(pt, ft)
        for pt, ft in zip(
            actual_ngram_candidates[f"pt_tpl_l{layer}"],
            actual_ngram_candidates[f"ft_tpl_l{layer}"],
        )
    ]

    actual_shift_table[f"euc_tpl_l{layer}"] = [
        euc_dist(pt, ft)
        for pt, ft in zip(
            actual_ngram_candidates[f"pt_tpl_l{layer}"],
            actual_ngram_candidates[f"ft_tpl_l{layer}"],
        )
    ]

actual_shift_table.sort_values(
    "score",
    ascending=False,
).head(20)
